In [1]:
import pandas as pd 
import altair as alt
import numpy as np

# Reiniciar

In [2]:
# Construcción de las variables descodificadas
import pandas as pd
import numpy as np

# 1. Cargar la base de datos
# Nota: Necesitas instalar 'pyreadstat' para leer archivos .sav (pip install pyreadstat)
file_path = 'SPSS/enemdu_persona_2026_01.sav'
df = pd.read_spss(file_path)
df['condact'].head()


0    Población Económicamente Inactiva
1    Población Económicamente Inactiva
2                Empleo Adecuado/Pleno
3                 Otro empleo no pleno
4                 Otro empleo no pleno
Name: condact, dtype: category
Categories (10, object): ['Desempleo abierto', 'Desempleo oculto', 'Empleo Adecuado/Pleno', 'Empleo no clasificado', ..., 'Otro empleo no pleno', 'Población Económicamente Inactiva', 'Subempleo por insuficiencia de ingresos', 'Subempleo por insuficiencia de tiempo de trab...]

In [3]:
# Reconstruir numeracion condact
mapa = {
"Empleo Adecuado/Pleno":1,
"Subempleo por insuficiencia de tiempo de trabajo":2,
"Subempleo por insuficiencia de ingresos":3,
"Otro empleo no pleno":4,
"Empleo no remunerado":5,
"Empleo no clasificado":6,
"Desempleo abierto":7,
"Desempleo oculto":8,
"Población Económicamente Inactiva":9
}

#diccionario tomado de la pagina del inec: https://anda.inec.gob.ec/anda/index.php/catalog/1048/datafile/F17/V1815?utm_source
df["condact"] = df["condact"].map(mapa)

In [4]:

# --- CONSTRUCCIÓN DE VARIABLES DE MERCADO LABORAL ---

# t_a: Población total
df['t_a'] = 1

# Definir p03, condact como categría ordinal
df["p03"] = pd.Categorical(df["p03"], ordered=True)

# Ajuste inicial de condact para menores de 15 años
# df["condact"] = df["condact"].cat.add_categories([0])
df.loc[df['p03'] < 15, 'condact'] = 0
print(df["condact"].value_counts())

# petn: Población en Edad de Trabajar (15 años o más)
df['petn'] = np.where(df['p03'] >= 15, 1, 0)

# pean: Población Económicamente Activa
df['pean'] = np.where(df['condact'].between(1, 8), 1, 0)

# empleo: Población con Empleo
df['empleo'] = np.where(df['condact'].between(1, 6), 1, 0)

# adec: Empleo Adecuado/Pleno
df['adec'] = np.where(df['condact'] == 1, 1, 0)

# sub: Subempleo
df['sub'] = np.where(df['condact'].between(2, 3), 1, 0)
df['sub_h'] = np.where(df['condact'] == 2, 1, 0)
df['sub_w'] = np.where(df['condact'] == 3, 1, 0)

# Otros tipos de empleo
df['oinad'] = np.where(df['condact'] == 4, 1, 0)
df['nr'] = np.where(df['condact'] == 5, 1, 0)
df['nc'] = np.where(df['condact'] == 6, 1, 0)

# Desempleo
df['desem'] = np.where(df['condact'].between(7, 8), 1, 0)
df['desemab'] = np.where(df['condact'] == 7, 1, 0)
df['desemoc'] = np.where(df['condact'] == 8, 1, 0)

# Desempleo Cesante y Nuevo (Basado en p37)
df['desem1'] = 0
df.loc[(df['condact'].between(7, 8)) & (df['p37'] == 1), 'desem1'] = 1

df['desem2'] = 0
df.loc[(df['condact'].between(7, 8)) & (df['p37'] == 2), 'desem2'] = 1

# pein: Población Económicamente Inactiva
df['pein'] = np.where(df['condact'] == 9, 1, 0)

# --- DESAGREGACIÓN DE LA SECEMP (Sectorización) ---

df['formal'] = np.where((df['secemp'] == 1) & (df['p03'] >= 15), 1, np.nan)
df['informal'] = np.where((df['secemp'] == 2) & (df['p03'] >= 15), 1, np.nan)
df['empdom'] = np.where((df['secemp'] == 3) & (df['p03'] >= 15), 1, np.nan)
df['nocla'] = np.where((df['secemp'] == 4) & (df['p03'] >= 15), 1, np.nan)

# Convertir ceros a NaN (equivalente a sysmis en SPSS) según tu script
vars_to_recode = [
    'petn', 'pean', 'empleo', 'adec', 'sub', 'sub_h', 'sub_w', 
    'oinad', 'nr', 'nc', 'desem', 'desemab', 'desemoc', 
    'desem1', 'desem2', 'pein'
]

for var in vars_to_recode:
    df[var] = df[var].replace(0, np.nan)

# POBLACIÓN DE 15 AÑOS Y MÁS
df['pobla15'] = np.where(df['p03'] >= 15, 1, np.nan)

print("Variables calculadas con éxito.")

condact
9.0    8351
1.0    5814
0.0    5404
4.0    3949
2.0    2262
5.0    1010
7.0     548
3.0      97
8.0      50
6.0      43
Name: count, dtype: int64
Variables calculadas con éxito.


In [5]:
for i in df.columns:
    print(f"Columna: {i}")
    #print(df[i].value_counts(dropna=False))
    print("-" * 50)

Columna: area
--------------------------------------------------
Columna: ciudad
--------------------------------------------------
Columna: conglomerado
--------------------------------------------------
Columna: panelm
--------------------------------------------------
Columna: vivienda
--------------------------------------------------
Columna: hogar
--------------------------------------------------
Columna: p01
--------------------------------------------------
Columna: p02
--------------------------------------------------
Columna: p03
--------------------------------------------------
Columna: p04
--------------------------------------------------
Columna: p05a
--------------------------------------------------
Columna: p05b
--------------------------------------------------
Columna: p06
--------------------------------------------------
Columna: p07
--------------------------------------------------
Columna: p08
--------------------------------------------------
Columna: p081
-

In [6]:
df['condact'].value_counts()

condact
9.0    8351
1.0    5814
0.0    5404
4.0    3949
2.0    2262
5.0    1010
7.0     548
3.0      97
8.0      50
6.0      43
Name: count, dtype: int64

In [7]:
# en p03 cambiamos " 98y más" por "98"
df['p03'] = df['p03'].replace(" 98 y más", "98")
df["p03"] = pd.to_numeric(df["p03"])

C:\Users\danli\AppData\Local\Temp\ipykernel_25952\3624392611.py:2: FutureWarning: The behavior of Series.replace (and DataFrame.replace) with CategoricalDtype is deprecated. In a future version, replace will only be used for cases that preserve the categories. To change the categories, use ser.cat.rename_categories instead.
  df['p03'] = df['p03'].replace(" 98 y más", "98")


In [8]:
df["p42"].unique()

[NaN, ' Empleado privado', ' Cuenta Propia', ' Empleado(a) Doméstico(a)', ' Trabajador del hogar no remunerado', ' Patrono', ' Empleado de gobierno', ' Jornalero o peón', ' Trabajador no del hogar no remunerado', ' Ayudante no remunerado de asalariado/jornalero']
Categories (9, object): [' Ayudante no remunerado de asalariado/jornalero', ' Cuenta Propia', ' Empleado de gobierno', ' Empleado privado', ..., ' Jornalero o peón', ' Patrono', ' Trabajador del hogar no remunerado', ' Trabajador no del hogar no remunerado']

In [9]:
# Recuperamos el codigo original de p42 según el diccionario del INEC
mapper_ocup = {
    " Empleado de gobierno": 1,
    " Empleado privado": 2,
    " Empleado tercerizado": 3,
    " Jornalero o peón": 4,
    " Patrono": 5,
    " Cuenta Propia": 6,
    " Trabajador del hogar no remunerado": 7,
    " Trabajador no del hogar no remunerado": 8,
    " Ayudante no remunerado de asalariado/jornalero": 9,
    " Empleado(a) Doméstico(a)": 10
}

df["p42"] = df["p42"].map(mapper_ocup)

In [10]:
df["p42"].value_counts()

p42
6     4837
2     4251
7     1306
4     1287
1     1266
10     372
5      356
8       23
9       13
Name: count, dtype: int64

In [11]:
# Recuperamos el codigo original de p42 según el diccionario del INEC
mapper_educ = {
    " Ninguno": 1,
    " Centro de alfabetización": 2,
    " Jardín de infantes": 3,
    " Primaria": 4,
    " Educación Básica": 5,
    " Secundaria": 6,
    " Educación Media": 7,
    " Superior no universitario": 8,
    " Superior Universitario": 9,
    " Post-grado": 10
}
df["p10a"] = df["p10a"].map(mapper_educ)

In [12]:
df['p10a'].value_counts()

p10a
6.0     5231
5.0     5175
4.0     5098
9.0     4541
8.0      808
10.0     788
1.0      778
2.0       18
Name: count, dtype: int64

In [13]:
df['p42']= pd.Categorical(df['p42'], ordered=True)

In [14]:
# --- 1. GENERACIÓN DE INDICADORES (TASAS BASE 100) ---
# Nota: np.where(condición_denominador, np.where(condición_numerador, 100, 0), np.nan)
# Esto asegura que si no pertenece al grupo base (ej. PEA), el resultado sea NaN y no 0.

# Tasas de Empleo y Adecuado
df['templeob'] = np.where(df['petn'] == 1, np.where(df['empleo'] == 1, 100, 0), np.nan)
df['templeog'] = np.where(df['pean'] == 1, np.where(df['empleo'] == 1, 100, 0), np.nan)
df['tadec']    = np.where(df['pean'] == 1, np.where(df['adec'] == 1, 100, 0), np.nan)

# Subempleo y sus variantes
df['tsub']     = np.where(df['pean'] == 1, np.where(df['sub'] == 1, 100, 0), np.nan)
df['tsub_h']   = np.where(df['pean'] == 1, np.where(df['sub_h'] == 1, 100, 0), np.nan)
df['tsub_w']   = np.where(df['pean'] == 1, np.where(df['sub_w'] == 1, 100, 0), np.nan)

# Otros indicadores de empleo
df['toinad']   = np.where(df['pean'] == 1, np.where(df['oinad'] == 1, 100, 0), np.nan)
df['tnr']      = np.where(df['pean'] == 1, np.where(df['nr'] == 1, 100, 0), np.nan)
df['tnc']      = np.where(df['pean'] == 1, np.where(df['nc'] == 1, 100, 0), np.nan)

# Desempleo y sus variantes
df['tdesem']   = np.where(df['pean'] == 1, np.where(df['desem'] == 1, 100, 0), np.nan)
df['tdesemab'] = np.where(df['pean'] == 1, np.where(df['desemab'] == 1, 100, 0), np.nan)
df['tdesemoc'] = np.where(df['pean'] == 1, np.where(df['desemoc'] == 1, 100, 0), np.nan)
df['tdesem1']  = np.where(df['pean'] == 1, np.where(df['desem1'] == 1, 100, 0), np.nan)
df['tdesem2']  = np.where(df['pean'] == 1, np.where(df['desem2'] == 1, 100, 0), np.nan)

# Tasas de participación
df['tpartib15'] = np.where(df['t_a'] == 1, np.where((df['pean'] == 1) & (df['p03'] >= 15), 100, 0), np.nan)
df['tpartig']   = np.where(df['petn'] == 1, np.where((df['pean'] == 1) & (df['p03'] >= 15), 100, 0), np.nan)

# Tasa de presión laboral (subempleo + desempleo)
df['tsubu'] = np.where(df['pean'] == 1, np.where((df['sub'] == 1) | (df['desem'] == 1), 100, 0), np.nan)

# --- 2. TIPO DE EMPLEADOR Y TRABAJO ---
# claempl: 1 Público, 2 Privado (incluye doméstico)
df['claempl'] = np.select([df['p42'] == 1, df['p42'].between(2, 10)], [1, 2], default=np.nan)

# asalind: 1 Asalariado, 2 Independiente
df['asalind'] = np.select([df['p42'].isin([1, 2, 3, 4, 10]), df['p42'].isin([5, 6])], [1, 2], default=np.nan)

# --- 3. NIVEL DE INSTRUCCIÓN ---
cond_instr = [
    (df['p10a'] == 1),
    (df['p10a'] == 2),
    (df['p10a'].isin([3, 4, 5])) | ((df['p10a'] == 6) & (df['p10b'] < 4)),
    (df['p10a'] == 7) | (df['p10a'] == 8) | ((df['p10a'] == 6) & (df['p10b'] > 3)),
    (df['p10a'].between(8, 10))
]
df['nnivins'] = np.select(cond_instr, [1, 2, 3, 4, 5], default=np.nan)

# --- 4. HORAS DE TRABAJO ---
# Reemplazar 999 (Missing en SPSS) por NaN
for col in ['p51a', 'p51b', 'p24']:
    df[col] = df[col].replace(999, np.nan)

df['ht'] = df[['p51a', 'p51b']].sum(axis=1, min_count=1)

# --- 5. ETNIA Y GRUPOS DE EDAD ---
# Etnia (Recodificación específica del script)
mapping_etnia = {1: 1, 2: 2, 3: 2, 4: 2, 5: 5, 6: 3, 7: 4, 8: 6}
df['etnia'] = df['p15'].map(mapping_etnia)

# Grupos de edad específicos (15-24, 25-34, etc.)
df['gedad'] = pd.cut(df['p03'], bins=[15, 25, 35, 45, 65, np.inf], 
                     labels=[1, 2, 3, 4, 5], right=False)

# --- 6. SECTORIZACIÓN ---
df['secto'] = np.nan
df.loc[df['formal'] == 1, 'secto'] = 1
df.loc[df['informal'] == 1, 'secto'] = 2
df.loc[df['empdom'] == 1, 'secto'] = 3
df.loc[df['nocla'] == 1, 'secto'] = 4

# Tasa de informalidad (Base 100)
df['tinformal'] = np.where(df['empleo'] == 1, np.where(df['informal'] == 1, 100, 0), np.nan)

# --- 7. FILTROS ÁREA (Urbano/Rural) ---
df['fil00'] = np.where(df['area'].isin([1, 2]), 1, np.nan)
df['fil01'] = np.where(df['area'] == 1, 1, np.nan) # Urbano
df['fil02'] = np.where(df['area'] == 2, 1, np.nan) # Rural

# --- 8. CARACTERIZACIÓN DESEMPLEO ---
df['desemb'] = np.select([df['desem1'] == 1, df['desem2'] == 1], [1, 2], default=np.nan)
df['desema'] = np.select([df['desemab'] == 1, df['desemoc'] == 1], [1, 2], default=np.nan)

In [15]:
df['p10a'].value_counts()

p10a
6.0     5231
5.0     5175
4.0     5098
9.0     4541
8.0      808
10.0     788
1.0      778
2.0       18
Name: count, dtype: int64

In [16]:
df['educacion']= df['p10a'].replace(mapper_educ)

In [17]:
df['educacion'].value_counts()

educacion
6.0     5231
5.0     5175
4.0     5098
9.0     4541
8.0      808
10.0     788
1.0      778
2.0       18
Name: count, dtype: int64

In [18]:
# Función auxiliar para obtener tabulados con factor de expansión
def tabulado_pesado(df, filas, columnas, valores='fexp', metrica='sum'):
    """Crea una tabla cruzada aplicando el factor de expansión."""
    tabla = df.pivot_table(index=filas, 
                           columns=columnas, 
                           values=valores, 
                           aggfunc=metrica)
    
    # Calcular porcentajes por columna (colpct en SPSS)
    tabla_pct = (tabla / tabla.sum()) * 100
    
    return tabla, tabla_pct


# --- 1. POBLACIÓN DE LOS PRINCIPALES INDICADORES ---

indicadores_pob = [
    't_a', 'pobla15', 'petn', 'pean', 'empleo', 'adec', 'sub', 
    'sub_h', 'sub_w', 'nr', 'oinad', 'nc', 'desem', 'desemab', 
    'desemoc', 'pein'
]

resumen_pob = {}

for ind in indicadores_pob:
    resumen_pob[ind] = df[df[ind] >= 1]['fexp'].sum()

poblacion_total = pd.Series(resumen_pob)

print("Población de Indicadores Principales:\n", poblacion_total)


# --- 2. TASAS DE LOS PRINCIPALES INDICADORES ---

tasas_cols = [
    'templeob', 'templeog', 'tadec', 'tsub', 'tsub_h', 'tsub_w', 
    'tnr', 'toinad', 'tnc', 'tdesem', 'tdesemab', 'tdesemoc', 
    'tpartig', 'tpartib15'
]

def calcular_tasa_pesada(df, tasa_col, factor='fexp'):
    df_sub = df.dropna(subset=[tasa_col])
    return (df_sub[tasa_col] * df_sub[factor]).sum() / df_sub[factor].sum()

print("\nTasas Nacionales (Pesadas):")

for t in tasas_cols:
    print(f"{t}: {calcular_tasa_pesada(df, t):.2f}%")


# --- 3. FUNCIÓN GENERAL DE CARACTERIZACIÓN ---

def caracterizar(df, variables, filtro=None, peso='fexp'):
    """
    Caracteriza múltiples variables con ponderación.

    df: DataFrame
    variables: lista de variables a analizar
    filtro: condición opcional (ej. df['empleo']==1)
    peso: factor de expansión
    """

    if filtro is not None:
        df = df[filtro]

    resultados = {}

    for var in variables:

        conteo = df.groupby(var)[peso].sum()

        porcentaje = (conteo / conteo.sum()) * 100

        tabla = pd.concat(
            [conteo, porcentaje],
            axis=1,
            keys=['Población', '%']
        )

        resultados[var] = tabla

    return resultados


# --- 3. CARACTERIZACIÓN (POBLACIÓN EMPLEADA) ---

variables_caracterizacion = [
    'p02',      # sexo
    'area',     # urbano/rural
    'gedad',   # grupo de edad
    'educacion',
    'etnia'
]

print("\nCaracterización de la Población Empleada:")

resultados = caracterizar(
    df,
    variables_caracterizacion,
    filtro=(df['empleo'] == 1)
)

for var, tabla in resultados.items():
    print(f"\n--- {var} ---")
    print(tabla)


# --- 4. SECTORIZACIÓN ---

tabla_sector, pct_sector = tabulado_pesado(
    df[df['empleo'] == 1],
    'secto',
    'area'
)

print("\nSectorización por Área (Porcentajes):\n", pct_sector)

Población de Indicadores Principales:
 t_a        1.900523e+07
pobla15    1.350685e+07
petn       1.350685e+07
pean       8.427512e+06
empleo     8.142909e+06
adec       3.085781e+06
sub        1.800361e+06
sub_h      1.754551e+06
sub_w      4.580943e+04
nr         6.110927e+05
oinad      2.628737e+06
nc         1.693779e+04
desem      2.846030e+05
desemab    2.536355e+05
desemoc    3.096749e+04
pein       5.079341e+06
dtype: float64

Tasas Nacionales (Pesadas):
templeob: 60.29%
templeog: 96.62%
tadec: 36.62%
tsub: 21.36%
tsub_h: 20.82%
tsub_w: 0.54%
tnr: 7.25%
toinad: 31.19%
tnc: 0.20%
tdesem: 3.38%
tdesemab: 3.01%
tdesemoc: 0.37%
tpartig: 62.39%
tpartib15: 44.34%

Caracterización de la Población Empleada:


C:\Users\danli\AppData\Local\Temp\ipykernel_25952\3213124704.py:70: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  conteo = df.groupby(var)[peso].sum()
C:\Users\danli\AppData\Local\Temp\ipykernel_25952\3213124704.py:4: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  tabla = df.pivot_table(index=filas,



--- p02 ---
           Población          %
p02                            
Hombre  4.927926e+06  60.518003
Mujer   3.214983e+06  39.481997

--- area ---
           Población          %
area                           
Rural   2.739616e+06  33.644193
Urbana  5.403293e+06  66.355807

--- gedad ---
          Población          %
gedad                         
1      1.092394e+06  13.415277
2      1.806429e+06  22.184071
3      2.048674e+06  25.158994
4      2.577121e+06  31.648651
5      6.182916e+05   7.593007

--- educacion ---
              Población          %
educacion                         
1.0        2.180760e+05   3.177980
2.0        9.058199e+03   0.132003
4.0        2.160406e+06  31.483176
5.0        4.711895e+05   6.866554
6.0        2.328447e+06  33.932011
8.0        2.754964e+05   4.014756
9.0        1.193208e+06  17.388394
10.0       2.062146e+05   3.005125

--- etnia ---
Empty DataFrame
Columns: [Población, %]
Index: []

Sectorización por Área (Porcentajes):
 Empty DataF

In [19]:
for i in df.columns:
    print(f"Columna: {i}")
    print(df[i].value_counts(dropna=False))
    print("-" * 50)

Columna: area
area
Urbana    20200
Rural      7328
Name: count, dtype: int64
--------------------------------------------------
Columna: ciudad
ciudad
90150.0     3026
170150.0    2633
70150.0     2205
10150.0     1908
180150.0    1609
            ... 
111152.0       9
150450.0       9
131750.0       8
60850.0        7
190251.0       7
Name: count, Length: 353, dtype: int64
--------------------------------------------------
Columna: conglomerado
conglomerado
900101    362
000102    271
900601    267
000101    262
000103    255
         ... 
165501      8
075002      8
011804      7
542803      7
000153      7
Name: count, Length: 902, dtype: int64
--------------------------------------------------
Columna: panelm
panelm
Panel D41    7033
Panel B31    6913
Panel A31    6882
Panel C42    6700
Name: count, dtype: int64
--------------------------------------------------
Columna: vivienda
vivienda
Vivienda Dos                  3627
Vivienda Seis                 3605
Vivienda Uno            

In [20]:
# Mapero de ciudadeds
ciudades = pd.read_excel('SPSS/Ciudades.xlsx')
ciudades.head()

,Value,Category
0,90150,Guayaquil
1,170150,Quito
2,130850,Manta
3,80150,Esmeraldas
4,110150,Loja


In [21]:
df['ciudad']=df['ciudad'].map(ciudades.set_index('Value')['Category'])

In [22]:
df_final = df[['ht','area','ciudad','etnia','condact','adec','desem',"p01", "p02", "p03", "p05b", "p06", "p07",'p10a','p32','p38','p42','p45']].rename(columns={
    "p01": "personas_hogar",
    "p02": "genero",
    "p03": "edad",
    "p05b": "seguro",
    "p06": "estado_civil",
    "p07": "asiste_a_clases",
    'condact': 'condicion_actividad',
    'adec': 'empleo_adecuado',
    'desem': 'desempleado',
    'p10a': 'nivel_educativo',
    'p45': 'tiempo_experiencia',
    'p32': 'como_busca_empleo',
    'p38': 'razon_dejo_trabajo',
    'p42': 'ocupacion_actual',
    'ht': 'horas_trabajo_totales'
})

In [23]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27528 entries, 0 to 27527
Data columns (total 18 columns):
 #   Column                 Non-Null Count  Dtype   
---  ------                 --------------  -----   
 0   horas_trabajo_totales  13297 non-null  float64 
 1   area                   27528 non-null  category
 2   ciudad                 27473 non-null  object  
 3   etnia                  0 non-null      float64 
 4   condicion_actividad    27528 non-null  float64 
 5   empleo_adecuado        5814 non-null   float64 
 6   desempleado            598 non-null    float64 
 7   personas_hogar         27528 non-null  category
 8   genero                 27528 non-null  category
 9   edad                   27528 non-null  float64 
 10  seguro                 27528 non-null  category
 11  estado_civil           23507 non-null  category
 12  asiste_a_clases        26174 non-null  category
 13  nivel_educativo        22437 non-null  float64 
 14  como_busca_empleo      12877 non-null 

In [24]:
df_final['ciudad'].value_counts()

ciudad
Guayaquil     3026
Quito         2633
Machala       2205
Cuenca        1908
Ambato        1609
              ... 
El tablon        9
El chaco         9
Pedernales       8
Pallatanga       7
Chito            7
Name: count, Length: 330, dtype: int64

In [25]:
df_final.to_csv('SPSS/ENEMDU_2026_01.csv', index=False)